<a href="https://colab.research.google.com/github/Santiago-Echeverri-Arteaga/Fisica_Computacional_2/blob/master/curso_2026_2/03_redes_fundamentos/33_perdidas_optimizadores_regularizacion.ipynb" target="_parent">
  <img src="https://colab.research.google.com/assets/colab-badge.svg"
       alt="Abrir en Colab"/>
</a>

# Pérdidas, optimizadores y regularización

**Pregunta guía:** ¿Qué objetivo optimizamos y qué sesgo introducimos?<br>
**Duración sugerida:** 4 horas.<br>
**Entorno:** CPU; datos incluidos o generados en memoria.

El orden de trabajo es siempre: problema → matemática → implementación
mínima → biblioteca → evaluación → interpretación física.


## La pérdida expresa una hipótesis de ruido

MSE corresponde a una observación gaussiana con varianza fija; MAE se
relaciona con Laplace y es menos sensible a valores extremos; Huber es
cuadrática cerca de cero y lineal lejos. Para clasificación, la
entropía cruzada es la log-verosimilitud negativa.


In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

error = np.linspace(-4, 4, 500)
delta = 1.0
pérdidas = {
    "MSE/2": 0.5*error**2,
    "MAE": np.abs(error),
    "Huber": np.where(np.abs(error)<=delta, 0.5*error**2, delta*(np.abs(error)-0.5*delta)),
}
for nombre, valores in pérdidas.items(): plt.plot(error, valores, label=nombre)
plt.xlabel("residuo"); plt.ylabel("pérdida"); plt.legend(); plt.show()

def bce_desde_logits(y, logits):
    # softplus(logit) - y*logit: estable incluso para logits grandes.
    return np.maximum(logits, 0) - y*logits + np.log1p(np.exp(-np.abs(logits)))

print(bce_desde_logits(np.array([0, 1]), np.array([-1000.0, 1000.0])))


## Optimizadores desde cero

Momentum acumula velocidad; RMSProp normaliza por una media de cuadrados;
Adam combina ambos y corrige el sesgo inicial. Los compararemos en la
función de Rosenbrock
$f(x,y)=(1-x)^2+100(y-x^2)^2$, cuyo valle curvo expone diferencias.


In [ ]:
def rosenbrock(theta):
    x, y = theta
    return (1-x)**2 + 100*(y-x**2)**2

def grad_rosenbrock(theta):
    x, y = theta
    return np.array([-2*(1-x)-400*x*(y-x**2), 200*(y-x**2)])

def optimizar(método, pasos=5_000):
    theta = np.array([-1.4, 1.5], dtype=float)
    m = np.zeros(2); v = np.zeros(2); trayectoria = [theta.copy()]
    for t in range(1, pasos+1):
        g = np.clip(grad_rosenbrock(theta), -1e3, 1e3)
        if método == "SGD":
            theta -= 0.001*g
        elif método == "Momentum":
            m = 0.9*m + g
            theta -= 0.0002*m
        elif método == "RMSProp":
            v = 0.99*v + 0.01*g*g
            theta -= 0.002*g/(np.sqrt(v)+1e-8)
        else:  # Adam
            m = 0.9*m + 0.1*g; v = 0.999*v + 0.001*g*g
            mh = m/(1-0.9**t); vh = v/(1-0.999**t)
            theta -= 0.003*mh/(np.sqrt(vh)+1e-8)
        if t % 20 == 0: trayectoria.append(theta.copy())
    return np.asarray(trayectoria), rosenbrock(theta)

filas = []
plt.figure(figsize=(7, 5))
for método in ["SGD", "Momentum", "RMSProp", "Adam"]:
    ruta, final = optimizar(método)
    filas.append({"optimizador": método, "pérdida_final": final, "x": ruta[-1,0], "y": ruta[-1,1]})
    plt.plot(ruta[:,0], ruta[:,1], label=método, alpha=0.8)
plt.scatter([1], [1], marker="*", s=180, c="gold", edgecolor="k", label="mínimo")
plt.xlabel("x"); plt.ylabel("y"); plt.legend(); plt.show()
display(pd.DataFrame(filas).set_index("optimizador"))


## Regularización

- L2 añade $\lambda\|W\|_2^2$ y favorece pesos pequeños.
- L1 añade $\lambda\|W\|_1$ y favorece esparsidad.
- Dropout multiplica activaciones por una máscara Bernoulli durante
  entrenamiento y usa escalado invertido $m/(1-p)$.
- Early stopping limita implícitamente cuánto se adapta el modelo.


In [ ]:
rng = np.random.default_rng(42)
activaciones = rng.normal(size=(10_000, 20))
for p in [0.0, 0.2, 0.5, 0.8]:
    máscara = rng.binomial(1, 1-p, size=activaciones.shape)
    salida = activaciones if p == 0 else activaciones*máscara/(1-p)
    print(f"p={p:.1f} | media={salida.mean():+.3f} | std={salida.std():.3f}")


**Ejercicios:** derive gradiente de L2; muestre por qué el escalado de
dropout conserva la esperanza; compare optimizadores con presupuesto
idéntico; diseñe una pérdida asimétrica para un sensor donde subestimar
sea más costoso que sobreestimar.
